In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import numpy as np
import pandas as pd
import xarray as xr
# import pingouin as pg
from os.path import join as pjoin
from tqdm.notebook import tqdm
import plotly.graph_objects as go
from scipy.stats import pearsonr, spearmanr, zscore
from natsort import natsorted
import itertools

sys.path.append('/home/austinbaggetta/csstorage3/CircleTrack/CircleTrackAnalysis')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import place_cells as pc
import plotting_functions as pf

In [ ]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6']
dpath = f'../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots'
chance_color = 'darkgrey'
avg_color = 'midnightblue'
subject_color = 'darkgrey'
ce_colors = ['darkgrey', 'midnightblue']
ce_color_dict = {'Control': 'darkgrey', 'Experimental': 'midnightblue'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
male_mice = ['mc44', 'mc46', 'mc54', 'mc55']
control_mice = ['mc46', 'mc49', 'mc52', 'mc54', 'mc59', 'mc60']
excluded_mice = ['mc47']
imaging5 = ['mc44', 'mc46', 'mc48', 'mc49', 'mc51', 'mc52']
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
control_renaming_dict = {'A1': 1, 'A2': 2, 'A3': 3, 'A4': 4, 'A5': 5,
                        'A6': 6, 'A7': 7, 'A8': 8, 'A9': 9, 'A10': 10,
                        'A11': 11, 'A12': 12, 'A13': 13, 'A14': 14, 'A15': 15,
                        'B1': 16, 'B2': 17, 'B3': 18, 'B4': 19, 'B5': 20}
experimental_renaming_dict = {'A1': 1, 'A2': 2, 'A3': 3, 'A4': 4, 'A5': 5,
                            'B1': 6, 'B2': 7, 'B3': 8, 'B4': 9, 'B5': 10,
                            'C1': 11, 'C2': 12, 'C3': 13, 'C4': 14, 'C5': 15,
                            'D1': 16, 'D2': 17, 'D3': 18, 'D4': 19, 'D5': 20}
bin_size = 0.1
velocity_thresh = 14
centroid_distance = 5
data_of_interest = 'aligned_minian' ## one of behav, aligned_minian, lin_behav

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

### Calculate the session-wide average population vector correlation between every session for every mouse.

In [ ]:
data_type = 'YrA'
data_dict = {'mouse': [], 'group': [], 'sex': [], 'sess_one': [], 'sess_two': [], 'r': [], 'p_value': []}
for experiment in experiment_folders:
    exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
    for mouse in tqdm(os.listdir(exp_path)):
        if mouse in excluded_mice:
            pass 
        else:
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')

            if mouse in imaging5:
                crossreg_path = pjoin(dpath, f'{experiment_folders[0]}/output/cross_registration_results')
                file_str = f'mappings_{centroid_distance}.pkl'
            else:
                crossreg_path = pjoin(dpath, f'{experiment_folders[1]}/output/cross_registration_results')
                file_str = f'mappings_{centroid_distance}_None.pkl'
            sex = 'Male' if mouse in male_mice else 'Female'
            group = 'Control' if mouse in control_mice else 'Experimental'
            mappings = pd.read_pickle(pjoin(crossreg_path, f'circletrack_data/{mouse}/{file_str}'))
            if mouse not in imaging5:
                try:
                    mappings = mappings.drop('2025_02_07', axis=1, level=1)
                except:
                    mappings = mappings
            mappings.columns = mappings.columns.droplevel(0)
            
            if mouse not in imaging5:
                mouse_files = os.listdir(mpath)[:-1]
            else:
                mouse_files = os.listdir(mpath)
                
            for d1, d2 in itertools.combinations_with_replacement(natsorted(mouse_files), r=2):
                sess_one = xr.open_dataset(pjoin(mpath, d1))[data_type]
                sess_two = xr.open_dataset(pjoin(mpath, d2))[data_type]

                if d1 == d2:
                    res = ctn.calculate_activity_correlation(sess_one, sess_two, test='pearson')
                else:
                    shared_cells = mappings[[sess_one.attrs['date'], sess_two.attrs['date']]].dropna().reset_index(drop=True)
                    shared_one = sess_one.sel(unit_id=shared_cells[sess_one.attrs['date']].values)
                    shared_two = sess_two.sel(unit_id=shared_cells[sess_two.attrs['date']].values)
                    res = ctn.calculate_activity_correlation(shared_one, shared_two, test='pearson')
                
                data_dict['mouse'].append(mouse)
                data_dict['group'].append(group)
                data_dict['sex'].append(sex)
                data_dict['sess_one'].append(sess_one.attrs['session_two'])
                data_dict['sess_two'].append(sess_two.attrs['session_two'])
                data_dict['r'].append(res[0])
                data_dict['p_value'].append(res[1])
correlation_df = pd.DataFrame(data_dict)

In [ ]:
## Plot the average population vector correlation heatmap for a single mouse below
mouse = 'mc48'
fig = pf.custom_graph_template(x_title='Day', y_title='Day', width=600, titles=[mouse])

renamed_cor = correlation_df.copy()
if mouse in control_mice:
        renamed_cor = renamed_cor.replace(control_renaming_dict)
else:
        renamed_cor = renamed_cor.replace(experimental_renaming_dict)

mdata = renamed_cor[renamed_cor['mouse'] == mouse]
matrix_one = mdata.pivot_table(index='sess_one', columns='sess_two', values='r')
matrix_one = matrix_one.fillna(0)
matrix_two = mdata.pivot_table(index='sess_two', columns='sess_one', values='r')
matrix_two = matrix_two.fillna(0)
matrix = matrix_one.values + matrix_two.values 
np.fill_diagonal(matrix, np.nan)

fig.add_trace(go.Heatmap(z=matrix, x=matrix_two.index, y=matrix_two.columns, coloraxis='coloraxis1'))
boundaries = [5.5, 10.5, 15.5]
for boundary in boundaries:
        fig.add_vline(x=boundary, line_width=1.5, line_color='red', opacity=1)
        fig.add_hline(y=boundary, line_width=1.5, line_color='red', opacity=1)
fig.update_layout(coloraxis_colorbar={'title': "Pearson's r"})
fig.update_coloraxes(colorscale='viridis')
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_avg_pvc_heatmap.png'))

In [ ]:
## Plot the average population vector correlation heatmap for control and experimental mice below
fig = pf.custom_graph_template(x_title='Day', y_title='Day', width=1000, titles=['Experimental', 'Control'], 
                               rows=1, columns=2, shared_y=True, shared_x=True)

renamed_cor = pd.DataFrame()
for mouse in correlation_df['mouse'].unique():
    mdata = correlation_df[correlation_df['mouse'] == mouse]
    if mouse in control_mice:
        mdata = mdata.replace(control_renaming_dict)
    else:
        mdata = mdata.replace(experimental_renaming_dict)
    renamed_cor = pd.concat([renamed_cor, mdata], ignore_index=True)

for idx, group in enumerate(renamed_cor['group'].unique()):
    print(group)
    gdata = renamed_cor[renamed_cor['group'] == group]
    avg_cor = gdata.groupby(['sess_one', 'sess_two'], as_index=False).agg({'r': 'mean'})
    gmatrix_one = avg_cor.pivot_table(index='sess_one', columns='sess_two', values='r')
    gmatrix_one = gmatrix_one.fillna(0)
    gmatrix_two = avg_cor.pivot_table(index='sess_two', columns='sess_one', values='r')
    gmatrix_two = gmatrix_two.fillna(0)
    gmatrix = gmatrix_one.values + gmatrix_two.values
    np.fill_diagonal(gmatrix, np.nan)
    fig.add_trace(go.Heatmap(z=gmatrix, x=gmatrix_two.index, y=gmatrix_two.columns, coloraxis='coloraxis1'), row=1, col=idx+1)

boundaries = [5.5, 10.5, 15.5]
for boundary in boundaries:
        fig.add_vline(x=boundary, line_width=1.5, line_color='red', opacity=1)
        fig.add_hline(y=boundary, line_width=1.5, line_color='red', opacity=1)
fig.update_layout(coloraxis_colorbar={'title': "Pearson's r"})
fig.update_coloraxes(colorscale='viridis')
fig.show()
fig.write_image(pjoin(fig_path, f'control_experimental_avg_pvc_heatmap.png'))

### Calculate average correlation for days in A compared to B, B compared to C, and C compared to D.

In [ ]:
res_dict = {'group': [], 'comparison': [], 'mean': [], 'sem': []}
compare_dict = {'A to B': ([6, 7, 8, 9, 10], [1, 2, 3, 4, 5]), 
                'B to C': ([11, 12, 13, 14, 15], [6, 7, 8, 9, 10]),
                'C to D': ([16, 17, 18, 19, 20], [11, 12, 13, 14, 15]),
                'A to C': ([11, 12, 13, 14, 15], [1, 2, 3, 4, 5]),
                'B to D': ([16, 17, 18, 19, 20], [6, 7, 8, 9, 10]),
                'A to D': ([16, 17, 18, 19, 20], [1, 2, 3, 4, 5])}
for comparison in compare_dict:
    days_of_interest_x = compare_dict[comparison][0]
    days_of_interest_y = compare_dict[comparison][1]
    avg_res = renamed_cor[(renamed_cor['sess_two'].isin(days_of_interest_x)) & (renamed_cor['sess_one'].isin(days_of_interest_y))].groupby(
    ['group'], as_index=False).agg({'r': ['mean', 'sem']})
    for group in avg_res['group']:
        gdata = avg_res[avg_res['group'] == group]
        res_dict['group'].append(gdata['group'].values[0])
        res_dict['comparison'].append(comparison)
        res_dict['mean'].append(gdata['r']['mean'].values[0])
        res_dict['sem'].append(gdata['r']['sem'].values[0])
comp_df = pd.DataFrame(res_dict)

In [ ]:
fig = pf.custom_graph_template(x_title='', y_title='Average Correlation')
for group in comp_df['group'].unique():
    gdata = comp_df[comp_df['group'] == group]
    fig.add_trace(go.Scatter(x=gdata['comparison'], y=gdata['mean'], mode='markers', marker_color=ce_color_dict[group],
                             error_y=dict(type='data', array=gdata['sem']), name=group))
fig.update_yaxes(range=[0, 1])
fig.show()
fig.write_image(pjoin(fig_path, 'avg_cor_between_contexts.png'))

### Calculate lick accuracy for each day to correlate with the correlation of average population activity on one day with another day.

In [ ]:
## Circle track behavior
lick_thresh = 5
data_of_interest = 'behav' ## one of behav, aligned_minian, lin_behav
circ_dict = {'mouse': [], 'experiment': [], 'sex': [], 'group': [], 'day': [], 'rewards': [], 'percent_correct': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            if mouse in excluded_mice:
                pass 
            else:
                mpath = pjoin(exp_path, mouse)
                sex = 'Male' if mouse in male_mice else 'Female'
                group = 'Control' if mouse in control_mice else 'Experimental'
                for idx, session in enumerate(os.listdir(mpath)):
                    behav = pd.read_feather(pjoin(mpath, session))
                    behav = behav[~behav['probe']] ## exclude probe
                    reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]    
                    pc = ctb.lick_accuracy(behav, port_list=[reward_one, reward_two], lick_threshold=lick_thresh, by_trials=False)
                    circ_dict['mouse'].append(mouse)
                    circ_dict['experiment'].append(behav['cohort'].unique()[0])
                    circ_dict['sex'].append(sex)
                    circ_dict['group'].append(group)
                    circ_dict['day'].append(idx+1)
                    circ_dict['rewards'].append(np.sum(behav['water']))
                    circ_dict['percent_correct'].append(pc)
ct_df = pd.DataFrame(circ_dict)

In [ ]:
## Select day of interest - first day of B for experimental mice, A6 for control
comparison_of_interset = {'x': 5, 'y': 6}
day_data = ct_df[ct_df['day'] == comparison_of_interset['y']].reset_index(drop=True)
pop_vec_cor = renamed_cor[(renamed_cor['sess_one'] == comparison_of_interset['x']) & (renamed_cor['sess_two'] == comparison_of_interset['y'])].reset_index(drop=True)
pop_vec_cor.loc[:, 'percent_correct'] = day_data['percent_correct']

In [ ]:
fig = pf.custom_graph_template(x_title="Pearson's R", y_title='', rows=1, columns=2, 
                               shared_x=True, shared_y=True, width=800)
for idx, group in enumerate(pop_vec_cor['group'].unique()):
    gdata = pop_vec_cor[pop_vec_cor['group'] == group]
    lm = pg.linear_regression(X=gdata['r'], y=gdata['percent_correct'], as_dataframe=False)
    fig.add_trace(go.Scattergl(x=gdata['r'], y=gdata['percent_correct'], name=group, legendgroup=group, 
                               mode='markers', marker_color=ce_color_dict[group]), row=1, col=idx+1)
    fig.add_trace(go.Scattergl(x=gdata['r'], y=lm['pred'], name=group, legendgroup=group, showlegend=False,
                               mode='lines', line_color=ce_color_dict[group]), row=1, col=idx+1)
fig.update_yaxes(title='Lick Accuracy (%)', range=[0, 100], col=1)
fig.show()

### Bin neural activity into x second bins and correlate activity between two sessions.

In [ ]:
time_bin = 30 ## in seconds
data_type = 'YrA'
data_dict = {'mouse': [], 'group': [], 'sex': [], 'sess_one': [], 'sess_two': [], 'time_bin': [], 'r': [], 'p_value': []}
for experiment in experiment_folders:
    exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
    # for mouse in tqdm(os.listdir(exp_path)):
    for mouse in ['mc48']:
        if mouse in excluded_mice:
            pass 
        else:
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')

            if mouse in imaging5:
                crossreg_path = pjoin(dpath, f'{experiment_folders[0]}/output/cross_registration_results')
                file_str = f'mappings_{centroid_distance}.pkl'
            else:
                crossreg_path = pjoin(dpath, f'{experiment_folders[1]}/output/cross_registration_results')
                file_str = f'mappings_{centroid_distance}_None.pkl'
            sex = 'Male' if mouse in male_mice else 'Female'
            group = 'Two-Context' if mouse in control_mice else 'Multi-Context'
            mappings = pd.read_pickle(pjoin(crossreg_path, f'circletrack_data/{mouse}/{file_str}'))
            if mouse not in imaging5:
                try:
                    mappings = mappings.drop('2025_02_07', axis=1, level=1)
                except:
                    mappings = mappings
            mappings.columns = mappings.columns.droplevel(0)
            
            if mouse not in imaging5:
                mouse_files = os.listdir(mpath)[:-1]
            else:
                mouse_files = os.listdir(mpath)
                
            for d1, d2 in itertools.combinations_with_replacement(natsorted(mouse_files), r=2):
                sess_one = xr.open_dataset(pjoin(mpath, d1))[data_type]
                sess_two = xr.open_dataset(pjoin(mpath, d2))[data_type]

                if d1 == d2:
                    binned_one = ctn.bin_in_time(sess_one, bin_size=time_bin)
                    binned_two = ctn.bin_in_time(sess_two, bin_size=time_bin)
                else:
                    shared_cells = mappings[[sess_one.attrs['date'], sess_two.attrs['date']]].dropna().reset_index(drop=True)
                    shared_one = sess_one.sel(unit_id=shared_cells[sess_one.attrs['date']].values)
                    shared_two = sess_two.sel(unit_id=shared_cells[sess_two.attrs['date']].values)
                    binned_one = ctn.bin_in_time(shared_one, bin_size=time_bin)
                    binned_two = ctn.bin_in_time(shared_two, bin_size=time_bin)
                
                for idx in np.arange(0, len(binned_one)):
                    if (binned_one[idx].shape[1] == 0) | (binned_two[idx].shape[1] == 0): ## if the imaging was cut short during the experiment
                        pass
                    else:
                        res = pearsonr(np.mean(binned_one[idx].values, axis=1), np.mean(binned_two[idx].values, axis=1))
                        data_dict['mouse'].append(mouse)
                        data_dict['group'].append(group)
                        data_dict['sex'].append(sex)
                        data_dict['sess_one'].append(sess_one.attrs['session_two'])
                        data_dict['sess_two'].append(sess_two.attrs['session_two'])
                        data_dict['time_bin'].append(time_bin*idx)
                        data_dict['r'].append(res[0])
                        data_dict['p_value'].append(res[1])
granular_cor_df = pd.DataFrame(data_dict)

In [ ]:
granular_cor_df = pd.DataFrame(data_dict)
granular_cor_df

In [ ]:
fig = pf.custom_graph_template(x_title='Time (s)', y_title="Pearson's Correlation", width=800)
for pair in [('A4', 'A5'), ('A5', 'B1')]:
    data = granular_cor_df[(granular_cor_df['sess_one'] == pair[0]) & (granular_cor_df['sess_two'] == pair[1])]
    fig.add_trace(go.Scattergl(x=data['time_bin'], y=data['r'], mode='lines+markers', name=f'{pair[0]} to {pair[1]}'))
fig.update_yaxes(range=[0, 1])
fig.show()

In [ ]:
fig = pf.custom_graph_template(x_title='Time (s)', y_title="Pearson's Correlation", width=800)
for pair in [('A1', 'A5'), ('B1', 'B5'), ('C1', 'C5'), ('D1', 'D5')]:
    data = granular_cor_df[(granular_cor_df['sess_one'] == pair[0]) & (granular_cor_df['sess_two'] == pair[1])]
    fig.add_trace(go.Scattergl(x=data['time_bin'], y=data['r'], mode='lines+markers', name=f'{pair[0]} to {pair[1]}'))
fig.update_yaxes(range=[0, 1])
fig.show()

### Correlate every time bin within a session to every other time bin within that session.

In [ ]:
time_bin = 30 ## in seconds
data_type = 'YrA'
data_dict = {'mouse': [], 'group': [], 'sex': [], 'session': [], 'bin_one': [], 'bin_two': [], 'r': [], 'p_value': []}
for experiment in experiment_folders:
    exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
    # for mouse in tqdm(os.listdir(exp_path)):
    for mouse in ['mc48']:
        if mouse in excluded_mice:
            pass 
        else:
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            sex = 'Male' if mouse in male_mice else 'Female'
            group = 'Two-Context' if mouse in control_mice else 'Multi-Context'
            
            if mouse not in imaging5:
                mouse_files = os.listdir(mpath)[:-1]
            else:
                mouse_files = os.listdir(mpath)
                
            for session in mouse_files:
                sess = xr.open_dataset(pjoin(mpath, session))[data_type]
                binned_sess = ctn.bin_in_time(sess, bin_size=time_bin)

                for b1, b2 in itertools.combinations_with_replacement(binned_sess, r=2):
                    if b1.shape[1] != b2.shape[1]:
                        min_frames = np.min((b1.shape[1], b2.shape[1]))
                        res = pearsonr(np.mean(b1[:, 0:min_frames].values, axis=1), np.mean(b2[:, 0:min_frames].values, axis=1))
                        data_dict['mouse'].append(mouse)
                        data_dict['group'].append(group)
                        data_dict['sex'].append(sex)
                        data_dict['session'].append(sess.attrs['session_two'])
                        data_dict['bin_one'].append(np.round(b1['behav_t'][-1].values))
                        data_dict['bin_two'].append(np.round(b2['behav_t'][-1].values))
                        data_dict['r'].append(res[0])
                        data_dict['p_value'].append(res[1])
                    else:
                        res = pearsonr(np.mean(b1.values, axis=1), np.mean(b2.values, axis=1))
                        data_dict['mouse'].append(mouse)
                        data_dict['group'].append(group)
                        data_dict['sex'].append(sex)
                        data_dict['session'].append(sess.attrs['session_two'])
                        data_dict['bin_one'].append(np.round(b1['behav_t'][-1].values))
                        data_dict['bin_two'].append(np.round(b2['behav_t'][-1].values))
                        data_dict['r'].append(res[0])
                        data_dict['p_value'].append(res[1])
within_sess_df = pd.DataFrame(data_dict)

In [ ]:
within_sess_df

### Correlate every time bin in one session with every time bin from another session.

In [ ]:
time_bin = 30 ## in seconds
data_type = 'YrA'
experiment = 'MultiCon_Imaging6'
mouse_list = ['mc56']
file_list = ['mc56_YrA_10.nc', 'mc56_YrA_11.nc']
data_dict = {'mouse': [], 'group': [], 'sex': [], 'session_one': [], 'session_two': [], 'bin_one': [], 'bin_two': [], 'r': [], 'p_value': []}

exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
for mouse in mouse_list:
    if mouse in excluded_mice:
        pass 
    else:
        mpath = pjoin(exp_path, f'{mouse}/{data_type}')

        if mouse in imaging5:
            crossreg_path = pjoin(dpath, f'{experiment_folders[0]}/output/cross_registration_results')
            file_str = f'mappings_{centroid_distance}.pkl'
        else:
            crossreg_path = pjoin(dpath, f'{experiment_folders[1]}/output/cross_registration_results')
            file_str = f'mappings_{centroid_distance}_None.pkl'
        sex = 'Male' if mouse in male_mice else 'Female'
        group = 'Two-Context' if mouse in control_mice else 'Multi-Context'
        mappings = pd.read_pickle(pjoin(crossreg_path, f'circletrack_data/{mouse}/{file_str}'))
        if mouse not in imaging5:
            try:
                mappings = mappings.drop('2025_02_07', axis=1, level=1)
            except:
                mappings = mappings
        mappings.columns = mappings.columns.droplevel(0)
        
        if mouse not in imaging5:
            mouse_files = os.listdir(mpath)[:-1]
        else:
            mouse_files = os.listdir(mpath)
        
        sess_one = xr.open_dataset(pjoin(mpath, file_list[0]))[data_type]
        sess_two = xr.open_dataset(pjoin(mpath, file_list[1]))[data_type]

        shared_cells = mappings[[sess_one.attrs['date'], sess_two.attrs['date']]].dropna().reset_index(drop=True)
        shared_one = sess_one.sel(unit_id=shared_cells[sess_one.attrs['date']].values)
        shared_two = sess_two.sel(unit_id=shared_cells[sess_two.attrs['date']].values)
        binned_one = ctn.bin_in_time(shared_one, bin_size=time_bin)
        binned_two = ctn.bin_in_time(shared_two, bin_size=time_bin)
            
        for idx1, idx2 in itertools.combinations_with_replacement(np.arange(0, len(binned_one)), r=2):
            if binned_one[idx1].shape[1] != binned_two[idx2].shape[1]:
                min_frames = np.min((binned_one[idx1].shape[1], binned_two[idx2].shape[1]))
                res = pearsonr(np.mean(binned_one[idx1][:, 0:min_frames].values, axis=1), np.mean(binned_two[idx2][:, 0:min_frames].values, axis=1))
                data_dict['mouse'].append(mouse)
                data_dict['group'].append(group)
                data_dict['sex'].append(sex)
                data_dict['session_one'].append(sess_one.attrs['session_two'])
                data_dict['session_two'].append(sess_two.attrs['session_two'])
                data_dict['bin_one'].append(time_bin*idx1)
                data_dict['bin_two'].append(time_bin*idx2)
                data_dict['r'].append(res[0])
                data_dict['p_value'].append(res[1])
            else:
                res = pearsonr(np.mean(binned_one[idx1].values, axis=1), np.mean(binned_two[idx2].values, axis=1))
                data_dict['mouse'].append(mouse)
                data_dict['group'].append(group)
                data_dict['sex'].append(sex)
                data_dict['session_one'].append(sess_one.attrs['session_two'])
                data_dict['session_two'].append(sess_two.attrs['session_two'])
                data_dict['bin_one'].append(time_bin*idx1)
                data_dict['bin_two'].append(time_bin*idx2)
                data_dict['r'].append(res[0])
                data_dict['p_value'].append(res[1])
between_sess_df = pd.DataFrame(data_dict)

## Create both halves of the heatmap
first_half = between_sess_df.pivot_table(values='r', index='bin_one', columns='bin_two')
first_half = first_half.fillna(0)
first = first_half.values ## must do this because when comparing between different sessions the diagonals are not 1
np.fill_diagonal(first, val=0)
second_half = between_sess_df.pivot_table(values='r', index='bin_two', columns='bin_one')
second_half = second_half.fillna(0)
combined = first + second_half.values

In [ ]:
s1 = f'{between_sess_df['session_one'].unique()[0]}'
s2 = f'{between_sess_df['session_two'].unique()[0]}'
fig = pf.custom_graph_template(x_title='Time (s)', y_title='Time (s)', 
                               titles=[f'{s1} to {s2}'])
fig.add_trace(go.Heatmap(x=np.unique(between_sess_df['bin_one']), y=np.unique(between_sess_df['bin_two']), z=combined, colorscale='viridis',
                         zmin=0, zmax=1))
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{s1}_{s2}_timebin_correlations_{time_bin}s.png'))

In [ ]:
s1 = f'{between_sess_df['session_one'].unique()[0]}'
s2 = f'{between_sess_df['session_two'].unique()[0]}'
fig = pf.custom_graph_template(x_title='Time (s)', y_title='Time (s)', 
                               titles=[f'{s1} to {s2}'])
fig.add_trace(go.Heatmap(x=np.unique(between_sess_df['bin_one']), y=np.unique(between_sess_df['bin_two']), z=combined, colorscale='viridis',
                         zmin=0, zmax=1))
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{s1}_{s2}_timebin_correlations_{time_bin}s.png'))

In [ ]:
s1 = f'{between_sess_df['session_one'].unique()[0]}'
s2 = f'{between_sess_df['session_two'].unique()[0]}'
fig = pf.custom_graph_template(x_title='Time (s)', y_title='Time (s)', 
                               titles=[f'{s1} to {s2}'])
fig.add_trace(go.Heatmap(x=np.unique(between_sess_df['bin_one']), y=np.unique(between_sess_df['bin_two']), z=combined, colorscale='viridis',
                         zmin=0, zmax=1))
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{s1}_{s2}_timebin_correlations_{time_bin}s.png'))

In [ ]:
s1 = f'{between_sess_df['session_one'].unique()[0]}'
s2 = f'{between_sess_df['session_two'].unique()[0]}'
fig = pf.custom_graph_template(x_title='Time (s)', y_title='Time (s)', 
                               titles=[f'{s1} to {s2}'])
fig.add_trace(go.Heatmap(x=np.unique(between_sess_df['bin_one']), y=np.unique(between_sess_df['bin_two']), z=combined, colorscale='viridis',
                         zmin=0, zmax=1))
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{s1}_{s2}_timebin_correlations_{time_bin}s.png'))

In [ ]:
s1 = f'{between_sess_df['session_one'].unique()[0]}'
s2 = f'{between_sess_df['session_two'].unique()[0]}'
fig = pf.custom_graph_template(x_title='Time (s)', y_title='Time (s)', 
                               titles=[f'{s1} to {s2}'])
fig.add_trace(go.Heatmap(x=np.unique(between_sess_df['bin_one']), y=np.unique(between_sess_df['bin_two']), z=combined, colorscale='viridis',
                         zmin=0, zmax=1))
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{s1}_{s2}_timebin_correlations_{time_bin}s.png'))

In [ ]:
s1 = f'{between_sess_df['session_one'].unique()[0]}'
s2 = f'{between_sess_df['session_two'].unique()[0]}'
fig = pf.custom_graph_template(x_title='Time (s)', y_title='Time (s)', 
                               titles=[f'{s1} to {s2}'])
fig.add_trace(go.Heatmap(x=np.unique(between_sess_df['bin_one']), y=np.unique(between_sess_df['bin_two']), z=combined, colorscale='viridis'))
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{s1}_{s2}_timebin_correlations_{time_bin}s.png'))

In [ ]:
s1 = f'{between_sess_df['session_one'].unique()[0]}'
s2 = f'{between_sess_df['session_two'].unique()[0]}'
fig = pf.custom_graph_template(x_title='Time (s)', y_title='Time (s)', 
                               titles=[f'{s1} to {s2}'])
fig.add_trace(go.Heatmap(x=np.unique(between_sess_df['bin_one']), y=np.unique(between_sess_df['bin_two']), z=combined, colorscale='viridis'))
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{s1}_{s2}_timebin_correlations_{time_bin}s.png'))